### 1. Load the graph

In [1]:
from pathlib import Path

import pandas as pd
import networkx as nx


PROCESSED_DATA_DIR = Path("../data/processed")

GRAPH_PATH = (
    PROCESSED_DATA_DIR
    / "story_similarity_graph.graphml"
)

G = nx.read_graphml(GRAPH_PATH)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 1266
Edges: 31491


### 2. Run Louvain

In [2]:
from networkx.algorithms.community import louvain_communities


louvain_communities_result = louvain_communities(
    G,
    weight="weight",
    resolution=1.0,
    seed=42
)

print(
    "Number of Louvain communities:",
    len(louvain_communities_result)
)

Number of Louvain communities: 19


### 3. Calculate Louvain modularity

In [3]:
from networkx.algorithms.community import modularity


louvain_modularity = modularity(
    G,
    louvain_communities_result,
    weight="weight"
)

print(
    "Louvain modularity:",
    round(louvain_modularity, 4)
)

Louvain modularity: 0.8656


### 4. Run greedy modularity

In [4]:
from networkx.algorithms.community import (
    greedy_modularity_communities
)


greedy_communities_result = (
    greedy_modularity_communities(
        G,
        weight="weight"
    )
)

print(
    "Number of greedy communities:",
    len(greedy_communities_result)
)

Number of greedy communities: 19


### 5. Calculate greedy modularity

In [5]:
greedy_modularity = modularity(
    G,
    greedy_communities_result,
    weight="weight"
)

print(
    "Greedy modularity:",
    round(greedy_modularity, 4)
)

Greedy modularity: 0.852


### 6. Compare the results

In [6]:
comparison = pd.DataFrame({
    "algorithm": [
        "Louvain",
        "Greedy modularity"
    ],
    "number_of_communities": [
        len(louvain_communities_result),
        len(greedy_communities_result)
    ],
    "modularity": [
        louvain_modularity,
        greedy_modularity
    ]
})

display(comparison)

,algorithm,number_of_communities,modularity
0,Louvain,19,0.865621
1,Greedy modularity,19,0.852049


### 7. Inspect community sizes

In [7]:
louvain_sizes = sorted(
    [
        len(community)
        for community in louvain_communities_result
    ],
    reverse=True
)

greedy_sizes = sorted(
    [
        len(community)
        for community in greedy_communities_result
    ],
    reverse=True
)

print("Largest Louvain communities:")
print(louvain_sizes[:10])

print("\nLargest greedy communities:")
print(greedy_sizes[:10])

Largest Louvain communities:
[252, 122, 96, 82, 78, 70, 67, 66, 62, 59]

Largest greedy communities:
[340, 116, 96, 74, 73, 71, 67, 61, 61, 55]


### 8. Assign each story to a community

In [8]:
louvain_assignments = {}

for community_id, community in enumerate(
    louvain_communities_result
):
    for story_id in community:
        louvain_assignments[story_id] = community_id


greedy_assignments = {}

for community_id, community in enumerate(
    greedy_communities_result
):
    for story_id in community:
        greedy_assignments[story_id] = community_id

In [9]:
community_assignments = pd.DataFrame({
    "story_id": list(G.nodes())
})

community_assignments["louvain_community"] = (
    community_assignments["story_id"]
    .map(louvain_assignments)
)

community_assignments["greedy_community"] = (
    community_assignments["story_id"]
    .map(greedy_assignments)
)

display(community_assignments.head())

,story_id,louvain_community,greedy_community
0,8193,18,0
1,2,0,17
2,16387,18,0
3,16410,1,2
4,8226,12,0


### 9. Save the results

In [10]:
OUTPUT_PATH = (
    PROCESSED_DATA_DIR
    / "community_assignments.csv"
)

community_assignments.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved community assignments to:")
print(OUTPUT_PATH)

Saved community assignments to:
..\data\processed\community_assignments.csv


## Community detection conclusion

Louvain and greedy modularity were applied to the weighted story similarity
graph.

Louvain was used as the main method, while greedy modularity served as a
baseline. The two methods were compared using modularity and the number and
sizes of detected communities.

The Louvain community assignments will be used in the next stage to identify
the keywords, places, stories, and narrators that characterize or connect the
communities.